In [28]:
# CELL 1
!pip install -q streamlit langchain-ollama langgraph pandas matplotlib seaborn

In [29]:
import subprocess
import time
import os

# 1. Install standard Linux hardware utilities (Helps Ollama detect the GPUs)
print("Configuring GPU environment...")
os.system("apt-get update && apt-get install -y pciutils")

# 2. Install Ollama
print("Installing Ollama...")
os.system("curl -fsSL https://ollama.com/install.sh | sh")

# Find the exact path
ollama_path = "/usr/local/bin/ollama"
if not os.path.exists(ollama_path):
    ollama_path = "/usr/bin/ollama" 

# 3. START SERVER WITH EXPLICIT GPU HOOKS
print("\nStarting Ollama server (GPU Enabled)...")
# Copy Kaggle's environment variables so Ollama can see the NVIDIA drivers
env = os.environ.copy()
# Force Ollama to see both Kaggle T4 GPUs
env["CUDA_VISIBLE_DEVICES"] = "0,1" 

ollama_process = subprocess.Popen(
    [ollama_path, "serve"], 
    env=env,  # <--- THIS IS THE MAGIC FIX
    stdout=subprocess.DEVNULL, 
    stderr=subprocess.DEVNULL
)

# Give it 5 seconds to boot up and connect to the GPUs
time.sleep(5)

# 4. Pull the Smart Coding Model
print("\nPulling qwen2.5-coder:7b (This is a 4.7GB download, taking ~2-3 minutes)...")
os.system(f"{ollama_path} pull qwen2.5-coder:7b")

print("\n✅ Ollama is hooked to the T4 GPUs and the Coder model is ready!")

Configuring GPU environment...
Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:5 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:6 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:8 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:9 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:10 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Hit:11 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Fetched 3,917 B in 1s (2,758 B/s)
Reading package lists...
Reading package lists...

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)



Building dependency tree...
Reading state information...
pciutils is already the newest version (1:3.7.0-6).
0 upgraded, 0 newly installed, 0 to remove and 168 not upgraded.
Installing Ollama...


>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> NVIDIA GPU installed.
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.



Starting Ollama server (GPU Enabled)...

Pulling qwen2.5-coder:7b (This is a 4.7GB download, taking ~2-3 minutes)...


pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ 


✅ Ollama is hooked to the T4 GPUs and the Coder model is ready!


pulling manifest ⠸ pulling manifest 
pulling 60e05f210007: 100% ▕██████████████████▏ 4.7 GB                         
pulling 66b9ea09bd5b: 100% ▕██████████████████▏   68 B                         
pulling 1e65450c3067: 100% ▕██████████████████▏ 1.6 KB                         
pulling 832dd9e00a68: 100% ▕██████████████████▏  11 KB                         
pulling d9bb33f27869: 100% ▕██████████████████▏  487 B                         
verifying sha256 digest 
writing manifest 
success 


In [30]:
%%writefile app_fixed.py
import streamlit as st
import pandas as pd
import os
import re
import matplotlib.pyplot as plt
import seaborn as sns
from langchain_ollama import ChatOllama
from langgraph.graph import StateGraph, END
from typing import TypedDict

# === UI STYLING ===
st.set_page_config(page_title="Local Data Oracle", layout="wide", initial_sidebar_state="expanded")

st.markdown("""
    <style>
    html, body, .stApp, .stMainBlockContainer { background-color: #ffffff !important; }
    h1, h2, h3, p, span, div, label { color: #0a0a0a !important; }
    .stChatMessage { background-color: #f8f9fa !important; border: 2px solid #e0e4e8 !important; border-radius: 12px !important; }
    [data-testid="chatMessageContainer"] [data-testid="stChatMessageContent"] { background-color: #e3f2fd !important; border-left: 4px solid #2196f3 !important; }
    [data-testid="chatMessageContainer"]:has(.stChatMessage:nth-child(2)) [data-testid="stChatMessageContent"] { background-color: #f1f8e9 !important; border-left: 4px solid #4caf50 !important; }
    </style>
""", unsafe_allow_html=True)

# === STATE MANAGEMENT ===
class AgentState(TypedDict):
    df: pd.DataFrame
    query: str
    code: str
    error: str
    retry_count: int
    output_text: str  
    final_report: str  # <--- NEW: Stores the ChatGPT-style summary
    has_visualization: bool  

# === AI CONNECTION ===
try:
    # Make sure this matches your 7B model!
    llm = ChatOllama(model="qwen2.5-coder:7b", temperature=0, base_url="http://localhost:11434")
except Exception as e:
    st.error(f"❌ Failed to connect to Ollama: {e}")
    st.stop()

# === NODE 1: THE ANALYST (Writes Code) ===
def analyst_node(state):
    cols = state['df'].columns.tolist()
    sample_data = state['df'].head(1).to_dict(orient='records')
    
    prompt = f"""
    You are an expert Python data analyst. 
    The DataFrame is ALREADY LOADED as 'df'. DO NOT use pd.read_csv().

    Dataset Columns: {cols}
    Sample Data (Row 1): {sample_data}
    
    User Request: {state['query']}
    Previous Error: {state['error']}
    
    RULES: 
    1. Write ONLY valid Python code using pandas.
    2. Create matplotlib visuals using plt.savefig('output.png') if asked. 
    3. Use print() to output numbers and dataframes so we can read them.
    4. Start with ```python and end with ```.
    """
    res = llm.invoke(prompt)
    
    try:
        match = re.search(r'```python\n(.*?)```', res.content, re.DOTALL)
        code = match.group(1).strip() if match else res.content.strip()
    except:
        code = res.content.strip()
        
    clean_code_lines = [line for line in code.split('\n') if "read_csv" not in line and "read_excel" not in line]
    final_code = '\n'.join(clean_code_lines)
    
    return {**state, "code": final_code, "retry_count": state['retry_count'] + 1}

# === NODE 2: THE EXECUTOR (Runs Code) ===
def executor_node(state):
    plt.clf()
    plt.close('all')
    import sys
    from io import StringIO
    
    old_stdout = sys.stdout
    sys.stdout = captured_output = StringIO()
    
    try:
        exec(state['code'], {}, {"df": state['df'], "pd": pd, "plt": plt, "sns": sns})
        output = captured_output.getvalue()
        sys.stdout = old_stdout
        has_viz = os.path.exists("output.png") and os.path.getsize("output.png") > 0
        
        return {**state, "error": "SUCCESS", "output_text": output or "Task completed silently.", "has_visualization": has_viz}
    except Exception as e:
        sys.stdout = old_stdout
        return {**state, "error": str(e), "output_text": "", "has_visualization": False}

# === NODE 3: THE REPORTER (Formats output like ChatGPT) ===
def reporter_node(state):
    prompt = f"""
    You are a Senior Data Science Consultant.
    
    The user asked this question: "{state['query']}"
    
    We ran a python script on their data and got this raw terminal output:
    ---
    {state['output_text']}
    ---
    
    Please write a polished, professional response for the user explaining these results. 
    Use formatting (bolding, bullet points) to make it easy to read. 
    Do NOT show the python code, just synthesize the insights.
    """
    res = llm.invoke(prompt)
    return {**state, "final_report": res.content}

# === GRAPH ROUTING ===
def should_retry(state):
    if state['error'] == "SUCCESS":
        return "reporter"  # Go to the new reporter node!
    elif state['retry_count'] >= 4:
        return "end"
    else:
        return "retry"

builder = StateGraph(AgentState)
builder.add_node("analyst", analyst_node)
builder.add_node("executor", executor_node)
builder.add_node("reporter", reporter_node) # Add Reporter

builder.set_entry_point("analyst")
builder.add_edge("analyst", "executor")
# Routing logic
builder.add_conditional_edges("executor", should_retry, {
    "retry": "analyst",
    "reporter": "reporter",
    "end": END
})
builder.add_edge("reporter", END)

agent = builder.compile()

# === UI LOGIC ===
st.title("🛡️ Local Data Oracle")

if "messages" not in st.session_state:
    st.session_state.messages = []
if "df" not in st.session_state:
    st.session_state.df = None

with st.sidebar:
    file = st.file_uploader("Upload CSV", type="csv")
    if file:
        st.session_state.df = pd.read_csv(file)
        st.success("Data Loaded!")

for m in st.session_state.messages:
    with st.chat_message(m["role"]):
        st.markdown(m["content"])
        if "img" in m and os.path.exists(m["img"]):
            st.image(m["img"])

if prompt := st.chat_input("Ask me to analyze your data..."):
    st.session_state.messages.append({"role": "user", "content": prompt})
    with st.chat_message("user"): st.markdown(prompt)
    
    if st.session_state.df is not None:
        with st.chat_message("assistant"):
            with st.status("Analyzing...", expanded=True) as status:
                st.write("🧠 Analyst writing code...")
                final_state = agent.invoke({"df": st.session_state.df, "query": prompt, "code": "", "error": "", "retry_count": 0, "output_text": "", "final_report": "", "has_visualization": False})
                
                st.write(f"🔄 Retries: {final_state['retry_count']}")
                with st.expander("View Python Code"):
                    st.code(final_state['code'], language='python')
                with st.expander("View Raw Data Output"):
                    st.text(final_state['output_text'])
                
                status.update(label="Analysis Complete!", state="complete")
            
            # Show the beautiful ChatGPT-style report instead of raw text
            if final_state['error'] == "SUCCESS":
                response = final_state['final_report']
            else:
                response = f"❌ Failed after {final_state['retry_count']} attempts.\n**Final Error:** {final_state['error']}"
                
            st.markdown(response)
            msg = {"role": "assistant", "content": response}
            
            if final_state['has_visualization'] and os.path.exists("output.png"):
                st.image("output.png")
                chart_name = f"chart_{len(st.session_state.messages)}.png"
                os.rename("output.png", chart_name)
                msg["img"] = chart_name
            st.session_state.messages.append(msg)

Overwriting app_fixed.py


In [31]:
import subprocess
import time
import os

# 1. Clear the deck
os.system("pkill -9 -f ollama")
time.sleep(2)

# 2. Start Ollama with a focus on the new model
ollama_path = "/usr/local/bin/ollama"
print("--- 🚀 Initializing Qwen 2.5 Coder 7B Engine ---")
subprocess.Popen([ollama_path, "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# 3. Wait for the server to wake up
time.sleep(10)

# 4. Pull the specific Qwen Coder model
# This is a 4.7GB download - Kaggle's internet is fast, but give it a moment
print("--- 📥 Pulling Qwen 2.5 Coder (This is a 4.7GB file) ---")
subprocess.run([ollama_path, "pull", "qwen2.5-coder:7b"])

# 5. Final Heartbeat check
!curl http://localhost:11434
print("\n✅ Qwen is loaded and ready to code!")

--- 🚀 Initializing Qwen 2.5 Coder 7B Engine ---
--- 📥 Pulling Qwen 2.5 Coder (This is a 4.7GB file) ---


pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ 

Ollama is running
✅ Qwen is loaded and ready to code!


pulling manifest ⠸ pulling manifest 
pulling 60e05f210007: 100% ▕██████████████████▏ 4.7 GB                         
pulling 66b9ea09bd5b: 100% ▕██████████████████▏   68 B                         
pulling 1e65450c3067: 100% ▕██████████████████▏ 1.6 KB                         
pulling 832dd9e00a68: 100% ▕██████████████████▏  11 KB                         
pulling d9bb33f27869: 100% ▕██████████████████▏  487 B                         
verifying sha256 digest 
writing manifest 
success 


In [ ]:
import subprocess
import sys
import time

# 1. Start Streamlit in the background
print("🚀 Starting Streamlit Server...")
subprocess.Popen(
    [sys.executable, "-m", "streamlit", "run", "app_fixed.py", "--server.port", "8501"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

# Give Streamlit 3 seconds to boot up
time.sleep(3)

# 2. Start Pinggy
print("\n=======================================================")
print("🌐 STARTING PINGGY TUNNEL")
print("Look for the URL ending in 'pinggy.link' in the output below.")
print("=======================================================\n")

# Use Pinggy via SSH to tunnel port 8501
!ssh -o StrictHostKeyChecking=no -p 443 -R0:localhost:8501 a.pinggy.io

🚀 Starting Streamlit Server...

🌐 STARTING PINGGY TUNNEL
Look for the URL ending in 'pinggy.link' in the output below.

Allocated port 4 for remote forward to localhost:8501
7=)0]8;;\                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                         ┌────────────────────────────┐                                                  │                            │                                     